## Cell 1: Setup
**What this demonstrates**: Environment initialisation for Step-Back RAG — Anthropic for step-back question generation and answer synthesis, OpenAI for embeddings, Chroma for dual retrieval.
**Expected output**: Setup confirmation with model names and retrieval configuration.

In [ ]:
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

import os
import time
import pathlib

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# Haiku generates the step-back abstraction cheaply; Sonnet synthesises the answer
STEPBACK_MODEL = 'claude-haiku-4-5-20251001'
ANSWER_MODEL   = 'claude-sonnet-4-6'
EMBED_MODEL    = 'text-embedding-3-small'
K_ABSTRACT     = 5   # chunks retrieved on the step-back (abstract) query
K_SPECIFIC     = 3   # chunks retrieved on the original (specific) query

client     = Anthropic()
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Step-back model  : {STEPBACK_MODEL}')
print(f'  Answer model     : {ANSWER_MODEL}')
print(f'  Abstract k       : {K_ABSTRACT} chunks')
print(f'  Specific k       : {K_SPECIFIC} chunks')

## Cell 2: Data — Basel III Excerpt
**What this demonstrates**: Loading the Basel III regulatory text — a document covering capital adequacy frameworks, minimum ratios, and buffer requirements. Specific queries about named provisions (e.g., CET1 for G-SIBs) often miss framework context that is indexed under more general headings; Step-Back RAG bridges this gap.
**Expected output**: Chunk count and a preview of the specificity problem Step-Back RAG addresses.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

# Basel III text: capital adequacy rationale, Pillar 1 minimums, buffer requirements,
# G-SIB surcharges, leverage ratio, liquidity standards
text = (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8')
raw_doc = Document(page_content=text, metadata={'source': 'basel_iii'})

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks: list[Document] = splitter.split_documents([raw_doc])

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='step_back_corpus',
)

print(f'Loaded: {len(text):,} characters → {len(chunks)} chunks')
print()
print('The specificity problem Step-Back RAG solves:')
print('  Specific query : "CET1 ratio for G-SIBs with surcharge"')
print('    → embedding anchors to G-SIB surcharge section')
print('    → misses the general CET1 framework that explains it')
print('  Step-back query: "What is CET1 and why does Basel III require it?"')
print('    → retrieves foundational capital adequacy context')
print('    → generator reasons correctly about the specific case')

## Cell 3: Core — Step-Back Generator, Dual Retrieval, Context Combiner, Pipeline
**What this demonstrates**: Three functions implement the full pattern: `generate_step_back_question` abstracts the specific query one level up, `step_back_rag` runs dual retrieval (abstract + specific), combines both contexts with clear labels, and generates the final answer.
**Expected output**: Function definitions confirmed.

In [ ]:
def generate_step_back_question(specific_query: str) -> str:
    """Rewrite a specific query as a higher-level, principle-level question.

    The step-back query is one level of abstraction up from the original —
    broad enough to retrieve foundational context, specific enough to still
    be meaningfully scoped (not maximally vague).

    Args:
        specific_query: The user's specific, detailed question.

    Returns:
        A broader, principle-level rephrasing of the query.
    """
    prompt = '\n'.join([
        'Rewrite the following question as a higher-level, more general question.',
        'The rewritten question should:',
        '  1. Ask about the general concept, rule, or framework the original is an instance of',
        '  2. Be ONE level of abstraction up — not maximally general',
        '  3. Still be meaningfully scoped to the same domain',
        'Return only the rewritten question. No explanation.',
        '',
        f'Question: {specific_query}',
    ])
    response = client.messages.create(
        model=STEPBACK_MODEL,
        max_tokens=150,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.content[0].text.strip()


def step_back_rag(query: str, k_abstract: int = K_ABSTRACT, k_specific: int = K_SPECIFIC) -> dict:
    """Full Step-Back RAG pipeline: abstract → dual retrieve → combine → generate.

    Runs two retrievals:
      1. On the step-back (abstract) query — retrieves general principle chunks
      2. On the original (specific) query   — retrieves case-specific chunks

    Both contexts are passed to the generator with section labels so the model
    can distinguish principle from specific evidence.

    Args:
        query:      The user's specific question.
        k_abstract: Chunks to retrieve on the abstract query.
        k_specific: Chunks to retrieve on the original query.

    Returns:
        dict with keys: query, step_back_query, abstract_docs, specific_docs,
                        combined_docs, answer, latency_ms.
    """
    t0 = time.perf_counter()

    # Step 1: generate the step-back (abstract) query
    step_back_query = generate_step_back_question(query)

    # Step 2: retrieve on abstract query — general principle context
    abstract_docs: list[Document] = vectorstore.similarity_search(
        step_back_query, k=k_abstract
    )

    # Step 3: retrieve on original query — specific case context
    specific_docs: list[Document] = vectorstore.similarity_search(
        query, k=k_specific
    )

    # Step 4: deduplicate across both sets before combining
    seen: set[str] = set()
    combined_docs: list[tuple[str, Document]] = []  # (label, doc)
    for label, docs in [('abstract', abstract_docs), ('specific', specific_docs)]:
        for doc in docs:
            key = doc.page_content[:80]
            if key not in seen:
                seen.add(key)
                combined_docs.append((label, doc))

    # Step 5: build labelled context — principle chunks precede specific chunks
    #         so the generator encounters the rule before the case
    abstract_context = '\n\n'.join(
        doc.page_content
        for label, doc in combined_docs if label == 'abstract'
    )
    specific_context = '\n\n'.join(
        doc.page_content
        for label, doc in combined_docs if label == 'specific'
    )
    combined_context = (
        f'--- General principle context ---\n{abstract_context}\n\n'
        f'--- Specific case context ---\n{specific_context}'
    )

    # Step 6: generate — system prompt instructs use of principle to reason about specific case
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=400,
        system=(
            'Answer using only the provided context. '
            'Use the general principle context to reason about the specific question. '
            'Cite specific figures or rules where available.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{combined_context}\n\nQuestion: {query}'}],
    )

    return {
        'query'          : query,
        'step_back_query': step_back_query,
        'abstract_docs'  : abstract_docs,
        'specific_docs'  : specific_docs,
        'combined_docs'  : combined_docs,
        'answer'         : response.content[0].text.strip(),
        'latency_ms'     : (time.perf_counter() - t0) * 1000,
    }


print('Step-Back RAG pipeline defined')
print('  generate_step_back_question() — abstract the specific query one level up')
print('  step_back_rag()               — abstract → dual retrieve → combine → generate')

## Cell 4: Run — CET1 Ratio for Systemically Important Banks
**What this demonstrates**: A specific Basel III query that benefits from step-back abstraction. The exact ratio for G-SIBs involves a surcharge on top of the base minimum — answering correctly requires understanding the general CET1 framework before the specific surcharge rule.
**Expected output**: Original query, generated step-back query, retrieval summary, and final answer citing both the general framework and the specific ratio.

In [ ]:
QUERY = 'What is the minimum Common Equity Tier 1 ratio for a systemically important bank?'

print(f'Specific query : {QUERY}')
print()

result = step_back_rag(QUERY)

print(f'Step-back query: {result["step_back_query"]}')
print()

# Retrieval summary
print(f'Retrieval:')
print(f'  Abstract path : {len(result["abstract_docs"])} chunks (general CET1 framework)')
print(f'  Specific path : {len(result["specific_docs"])} chunks (G-SIB / surcharge context)')
print(f'  Combined      : {len(result["combined_docs"])} unique chunks after dedup')
print(f'  Latency       : {result["latency_ms"]:.0f} ms')
print()

print('=' * 65)
print('ANSWER')
print('=' * 65)
print(result['answer'])

## Cell 5: Inspect — Step-Back Query, Per-Path Contexts, Combined Context
**What this demonstrates**: The mechanics of dual retrieval — what each path retrieved, how many chunks overlapped, and what the combined labelled context looks like before it reaches the generator. Also shows how single-query retrieval on the specific question alone would have differed.
**Expected output**: Per-path chunk previews, combined context structure, and a baseline comparison.

In [ ]:
# ── Step-back query comparison ─────────────────────────────────────────────────
print('QUERY ABSTRACTION')
print('=' * 65)
print(f'  Original  : {result["query"]}')
print(f'  Step-back : {result["step_back_query"]}')
print()

# ── Abstract path chunks ───────────────────────────────────────────────────────
print('ABSTRACT PATH — general principle context')
print('=' * 65)
for i, doc in enumerate(result['abstract_docs'], 1):
    preview = doc.page_content[:90].replace('\n', ' ')
    print(f'  {i}. {preview}...')

# ── Specific path chunks ───────────────────────────────────────────────────────
print()
print('SPECIFIC PATH — case-specific context')
print('=' * 65)
for i, doc in enumerate(result['specific_docs'], 1):
    preview = doc.page_content[:90].replace('\n', ' ')
    print(f'  {i}. {preview}...')

# ── Overlap analysis ──────────────────────────────────────────────────────────
abstract_keys = {d.page_content[:80] for d in result['abstract_docs']}
specific_keys = {d.page_content[:80] for d in result['specific_docs']}
overlap       = abstract_keys & specific_keys
print()
print('OVERLAP ANALYSIS')
print('=' * 65)
print(f'  Abstract-only chunks : {len(abstract_keys - specific_keys)}')
print(f'  Specific-only chunks : {len(specific_keys - abstract_keys)}')
print(f'  Overlap (deduped)    : {len(overlap)}')
print(f'  Combined unique      : {len(result["combined_docs"])}')

# ── Labelled context preview ───────────────────────────────────────────────────
print()
print('COMBINED CONTEXT STRUCTURE (first 200 chars per section)')
print('=' * 65)
abstract_chunks = [doc for label, doc in result['combined_docs'] if label == 'abstract']
specific_chunks = [doc for label, doc in result['combined_docs'] if label == 'specific']
print(f'  [General principle — {len(abstract_chunks)} chunks]')
if abstract_chunks:
    print(f'  {abstract_chunks[0].page_content[:200].replace(chr(10), " ")}...')
print(f'  [Specific case — {len(specific_chunks)} chunks]')
if specific_chunks:
    print(f'  {specific_chunks[0].page_content[:200].replace(chr(10), " ")}...')

# ── Baseline comparison ────────────────────────────────────────────────────────
print()
print('BASELINE: single-query retrieval on specific question only')
print('=' * 65)
baseline_docs = vectorstore.similarity_search(QUERY, k=5)
baseline_keys = {d.page_content[:80] for d in baseline_docs}
stepback_keys = {d.page_content[:80] for label, d in result['combined_docs']}
new_in_sb     = stepback_keys - baseline_keys
print(f'  Baseline top-5       : {len(baseline_keys)} chunks')
print(f'  Step-Back combined   : {len(stepback_keys)} chunks')
print(f'  New via step-back    : {len(new_in_sb)} principle chunks baseline missed')

## Cell 6: Fintech — Specific Regulatory Case with General Principle Context
**What this demonstrates**: Step-Back RAG on a specific Tier 2 capital eligibility question. The query names a particular instrument configuration; the step-back abstracts to the general Tier 2 eligibility framework. Combining both contexts gives the generator the rule it needs to evaluate the specific case correctly.
**Expected output**: Step-back abstraction, dual retrieval results, and an answer that explicitly applies the general framework to the specific instrument case.

In [ ]:
FINTECH_QUERY = (
    'What capital conservation buffer requirements apply '
    'when a bank\'s CET1 ratio falls below the regulatory minimum?'
)

print('FINTECH: SPECIFIC REGULATORY CASE — GENERAL PRINCIPLE CONTEXT')
print(f'Specific query: {FINTECH_QUERY}')
print()
print('Why Step-Back helps:')
print('  The query asks about a specific trigger condition for the buffer.')
print('  To answer correctly the model needs:')
print('    1. General: what is the capital conservation buffer and why does it exist?')
print('    2. Specific: what restrictions apply at each CET1 threshold?')
print('  Single-query retrieval on the specific phrasing may retrieve only')
print('  restriction tables without the framework context that explains them.')
print()

fintech_result = step_back_rag(FINTECH_QUERY)

print(f'Step-back query: {fintech_result["step_back_query"]}')
print()

# Per-path retrieval
print('Abstract path (framework context):')
for i, doc in enumerate(fintech_result['abstract_docs'][:3], 1):
    preview = doc.page_content[:80].replace('\n', ' ')
    print(f'  {i}. {preview}...')

print()
print('Specific path (trigger condition context):')
for i, doc in enumerate(fintech_result['specific_docs'], 1):
    preview = doc.page_content[:80].replace('\n', ' ')
    print(f'  {i}. {preview}...')

print()
print(f'Combined: {len(fintech_result["combined_docs"])} unique chunks')
print(f'Latency : {fintech_result["latency_ms"]:.0f} ms')
print()
print('=' * 65)
print('REGULATORY ANSWER')
print('=' * 65)
print(fintech_result['answer'])
print()
print('-' * 65)
print('Step-Back RAG value for regulatory queries:')
print('  Abstract path surfaces the buffer framework and rationale')
print('  Specific path surfaces the threshold-linked restriction table')
print('  Generator applies framework reasoning to the specific trigger')